In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import numpy as np
import pandas as pd

%matplotlib inline
import matplotlib.pyplot as plt

from sklearn.preprocessing import OneHotEncoder, StandardScaler

from config import config
import utils
import objects.datasets

In [10]:
config

{'general': {'project_name': 'Titanic and Houses', 'data_installed': True}, 'paths': {'titanic_targets': 'objects/titanic/gender_submission.csv', 'titanic_train': 'objects/titanic/train.csv', 'titanic_test': 'objects/titanic/test.csv'}}

## Titanic

### Обзор датасета

In [5]:
df_train = pd.read_csv(config.paths.titanic_train)
df_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 118.9 KB


In [12]:
df_test = pd.read_csv(config.paths.titanic_test)
df_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Pclass       418 non-null    int64  
 2   Name         418 non-null    str    
 3   Sex          418 non-null    str    
 4   Age          332 non-null    float64
 5   SibSp        418 non-null    int64  
 6   Parch        418 non-null    int64  
 7   Ticket       418 non-null    str    
 8   Fare         417 non-null    float64
 9   Cabin        91 non-null     str    
 10  Embarked     418 non-null    str    
dtypes: float64(2), int64(4), str(5)
memory usage: 52.8 KB


### Очистка

In [13]:
df_train.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [14]:
df_test.isna().sum()

PassengerId      0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64

#### Пустые значения в Age

In [15]:
df_train['Initial'] = df_train['Name'].str.extract('([A-Za-z]+)\.')
pd.crosstab(df_train['Initial'], df_train['Sex']).T

Initial,Capt,Col,Countess,Don,Dr,Jonkheer,Lady,Major,Master,Miss,Mlle,Mme,Mr,Mrs,Ms,Rev,Sir
Sex,,,,,,,,,,,,,,,,,
female,0,0,1,0,1,0,1,0,0,182,2,1,0,125,1,0,0
male,1,2,0,1,6,1,0,2,40,0,0,0,517,0,0,6,1


In [16]:
unique_initials = df_train['Initial'].unique().tolist()
initials_to_replace = []
for unique_initial in unique_initials:
    if unique_initial in ['Mlle', 'Mme', 'Ms', 'Dr', 'Major', 'Lady', 'Capt', 'Sir', 'Don', 'Dona']:
        if unique_initial in ['Mlle', 'Mme', 'Ms']:
            initials_to_replace.append('Miss')
        elif unique_initial in ['Dr', 'Major', 'Capt', 'Sir', 'Don']:   
            initials_to_replace.append('Mr')
        else:
            initials_to_replace.append('Mrs')
    elif unique_initial in ['Mr', 'Mrs', 'Miss', 'Master']:
        initials_to_replace.append(unique_initial)
    else:
        initials_to_replace.append('Other')

df_train['Initial'] = df_train['Initial'].replace(unique_initials, initials_to_replace)
pd.crosstab(df_train['Initial'], df_train['Sex']).T

Initial,Master,Miss,Mr,Mrs,Other
Sex,,,,,
female,0,186,1,126,1
male,40,0,528,0,9


In [17]:
mean_age_by_initial = df_train.groupby('Initial')['Age'].mean()

for initial in mean_age_by_initial.keys():
    df_train.loc[(df_train['Age'].isnull()) & (df_train['Initial'] == initial), 'Age'] = mean_age_by_initial[initial]

In [18]:
df_train['Age'].isnull().sum()

np.int64(0)

**Embarked**

In [19]:
df_train['Embarked'] = df_train['Embarked'].fillna('S')
df_train['Embarked'].isna().sum()

np.int64(0)

**Fare (только для test)**

In [20]:
df_test[df_test['Fare'].isna() == True]

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
152,1044,3,"Storey, Mr. Thomas",male,60.5,0,0,3701,NaN,NaN,S


In [21]:
df_test.loc[(df_test['Fare'].isnull()), 'Fare'] = df_test.groupby('Pclass')['Fare'].mean()[3]

In [22]:
df_test.isna().sum()

PassengerId      0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          327
Embarked         0
dtype: int64

### Feature engineering

*Family size* feature

In [23]:
df_train['Family_size'] = df_train['Parch'] + df_train['SibSp']

In [24]:
df_train = df_train.drop(columns=['Parch', 'SibSp', 'Name', 'Ticket', 'Cabin', 'PassengerId'])
df_train.head()

,Survived,Pclass,Sex,Age,Fare,Embarked,Initial,Family_size
0,0,3,male,22.0,7.2500,S,Mr,1
1,1,1,female,38.0,71.2833,C,Mrs,1
2,1,3,female,26.0,7.9250,S,Miss,0
3,1,1,female,35.0,53.1000,S,Mrs,1
4,0,3,male,35.0,8.0500,S,Mr,0


In [32]:
df_test['Pclass'].name

'Pclass'

In [44]:
pclass_encoder = OneHotEncoder(sparse_output=False)
pclass_encoded = pclass_encoder.fit_transform(df_train['Pclass'].to_numpy().reshape(-1, 1))
df_pclass = pd.DataFrame(pclass_encoded, columns=[f'Pclass_{i}' for i in reversed(df_test['Pclass'].unique())])
df_pclass

,Pclass_1,Pclass_2,Pclass_3
0,0.0,0.0,1.0
1,1.0,0.0,0.0
2,0.0,0.0,1.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
886,0.0,1.0,0.0
887,1.0,0.0,0.0
888,0.0,0.0,1.0
889,1.0,0.0,0.0


In [50]:
df_pclass_encoded = utils.make_one_hot_encoding(df_train['Pclass'], drop_first=True)

In [51]:
df_embarked_encoded = utils.make_one_hot_encoding(df_train['Embarked'], drop_first=True)

In [65]:
df_initial_encoded = utils.make_one_hot_encoding(df_train['Initial'], drop_first=True)
df_initial_encoded

,Initial_Miss,Initial_Mr,Initial_Mrs,Initial_Other
0,0.0,1.0,0.0,0.0
1,0.0,0.0,1.0,0.0
2,1.0,0.0,0.0,0.0
3,0.0,0.0,1.0,0.0
4,0.0,1.0,0.0,0.0
...,...,...,...,...
886,0.0,0.0,0.0,1.0
887,1.0,0.0,0.0,0.0
888,1.0,0.0,0.0,0.0
889,0.0,1.0,0.0,0.0


In [54]:
df_family_size_encoded = utils.make_one_hot_encoding(df_train['Family_size'], drop_first=True)

In [35]:
df_train['Sex'] = df_train['Sex'].replace(['male', 'female'], [0, 1])

In [62]:
df_age_scaled = utils.make_standard_scaling(df_train['Age'])
df_age_scaled

,Age
0,-0.587617
1,0.617832
2,-0.286255
3,0.391810
4,0.391810
...,...
886,-0.210914
887,-0.813639
888,-0.598165
889,-0.286255


In [63]:
df_fare_scaled = utils.make_standard_scaling(df_train['Fare'])

In [66]:
concat_dataframe = pd.concat([df_age_scaled, df_fare_scaled, df_initial_encoded, df_embarked_encoded, df_pclass_encoded, df_family_size_encoded], axis=1)
concat_dataframe.insert(0, 'Sex', df_train['Sex'])
concat_dataframe['Survived'] = df_train['Survived']
concat_dataframe

,Sex,Age,Fare,Initial_Miss,Initial_Mr,Initial_Mrs,Initial_Other,Embarked_Q,Embarked_S,Pclass_2,Pclass_3,Family_size_1,Family_size_2,Family_size_3,Family_size_4,Family_size_5,Family_size_6,Family_size_7,Family_size_10,Survived
0,0,-0.587617,-0.502445,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,1,0.617832,0.786845,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
2,1,-0.286255,-0.488854,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
3,1,0.391810,0.420730,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
4,0,0.391810,-0.486337,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,-0.210914,-0.386671,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
887,1,-0.813639,-0.044381,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
888,1,-0.598165,-0.176263,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0
889,0,-0.286255,-0.044381,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1


In [6]:
type(df_train['Sex'])

pandas.Series

In [38]:
df_preparer = objects.datasets.TitanicDatasetPrepare(config.paths.titanic_train)
df_check = df_preparer.prepare_dataset()
statistics = df_preparer.statistics
# df_check.head(10)
X, y = df_preparer.to_xy()
y

array([0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1,
       1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1,
       1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1,
       1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0,
       1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0,
       0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0,
       0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0,
       1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0,
       1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1,
       0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0,
       0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0,
       1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1,

In [34]:
df_preparer = objects.datasets.TitanicDatasetPrepare(config.paths.titanic_test)
df_check = df_preparer.prepare_dataset(statistics)
df_check.head(10)

,Sex,Age,Fare,Family_size,Pclass_2,Pclass_3,Embarked_Q,Embarked_S,Initial_Miss,Initial_Mr,Initial_Mrs,Initial_Other
0,0,0.353941,-0.490508,-0.560660,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
1,1,1.295169,-0.507194,0.059127,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
2,0,2.424644,-0.453112,-0.560660,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
3,0,-0.210796,-0.473739,-0.560660,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
4,1,-0.587287,-0.400792,0.678913,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
5,0,-1.189673,-0.462419,-0.560660,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
6,1,0.015099,-0.494532,-0.560660,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0
7,0,-0.286094,-0.064480,0.678913,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
8,1,-0.888480,-0.502582,-0.560660,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
9,0,-0.662586,-0.162078,0.678913,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0


In [36]:
X = df_preparer.to_xy()
X[:10]

array([[ 0.        ,  0.35394111, -0.49050767, -0.56065994,  0.        ,
         1.        ,  1.        ,  0.        ,  0.        ,  1.        ,
         0.        ,  0.        ],
       [ 1.        ,  1.29516947, -0.50719398,  0.05912667,  0.        ,
         1.        ,  0.        ,  1.        ,  0.        ,  0.        ,
         1.        ,  0.        ],
       [ 0.        ,  2.42464351, -0.45311239, -0.56065994,  1.        ,
         0.        ,  1.        ,  0.        ,  0.        ,  1.        ,
         0.        ,  0.        ],
       [ 0.        , -0.21079592, -0.47373886, -0.56065994,  0.        ,
         1.        ,  0.        ,  1.        ,  0.        ,  1.        ,
         0.        ,  0.        ],
       [ 1.        , -0.58728726, -0.40079158,  0.67891328,  0.        ,
         1.        ,  0.        ,  1.        ,  0.        ,  0.        ,
         1.        ,  0.        ],
       [ 0.        , -1.18967342, -0.46241945, -0.56065994,  0.        ,
         1.        ,  

### Training

In [5]:
import numpy as np
from sklearn.model_selection import StratifiedKFold
 
X = np.array([[1, 2], [3, 4], [1, 2], [3, 4]])
y = np.array([0, 0, 1, 1])
skf = StratifiedKFold(n_splits=2)
 
for train_index, test_index in skf.split(X, y):
    print("TRAIN:", train_index, "TEST:", test_index)
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
'''
result:
TRAIN: [1 3] TEST: [0 2]
TRAIN: [0 2] TEST: [1 3]
'''

TRAIN: [1 3] TEST: [0 2]
TRAIN: [0 2] TEST: [1 3]


'\nresult:\nTRAIN: [1 3] TEST: [0 2]\nTRAIN: [0 2] TEST: [1 3]\n'

In [2]:
preparer = objects.datasets.TitanicDatasetPrepare(config.paths.titanic_train)
df = preparer.prepare_dataset()
df

,Survived,Sex,Age,Fare,Family_size,Pclass_2,Pclass_3,Embarked_Q,Embarked_S,Initial_Miss,Initial_Mr,Initial_Mrs,Initial_Other
0,0,0,-0.587617,-0.502445,0.059160,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
1,1,1,0.617832,0.786845,0.059160,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,1,1,-0.286255,-0.488854,-0.560975,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0
3,1,1,0.391810,0.420730,0.059160,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
4,0,0,0.391810,-0.486337,-0.560975,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,0,-0.210914,-0.386671,-0.560975,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
887,1,1,-0.813639,-0.044381,-0.560975,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
888,0,1,-0.598165,-0.176263,1.299429,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0
889,1,0,-0.286255,-0.044381,-0.560975,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [ ]:
X, y = preparer.to_xy()

array([0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1,
       1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1,
       1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1,
       1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0,
       1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0,
       0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0,
       0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0,
       1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0,
       1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1,
       0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0,
       0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0,
       1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1,

In [15]:
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [27]:
preparer = objects.datasets.TitanicDatasetPrepare(config.paths.titanic_train)
df = preparer.prepare_dataset()
X, y = preparer.to_xy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=config.general.random_state, stratify=y, )

skf = StratifiedKFold(n_splits=config.cv.k_forlds, shuffle=config.cv.shuffle, random_state=config.general.random_state)
models = []
metrics = []

for train_index, val_index in skf.split(X_train, y_train):
    model = LogisticRegression(random_state=config.general.random_state, )

    X_train, X_val = X[train_index], X[val_index]
    y_train, y_val = y[train_index], y[val_index]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)

    metrics.append({'Accuracy': accuracy_score(y_val, y_pred), 'Precision': precision_score(y_val, y_pred), 'Recal': recall_score(y_val, y_pred), 'F1': f1_score(y_val, y_pred)})
    models.append(model)

metrics_df = pd.DataFrame(metrics)
metrics_avg = metrics_df.to_numpy().mean(axis=0)
metrics_avg = pd.DataFrame(metrics_df.to_numpy().mean(axis=0).reshape(1, -1), columns=metrics_df.columns)
metrics_avg

,Accuracy,Precision,Recal,F1
0,0.811859,0.790369,0.712451,0.746389


In [20]:
metrics_df = pd.DataFrame(metrics)
metrics_df.head()

,Accuracy,Precision,Recal,F1
0,0.776224,0.648148,0.729167,0.686275
1,0.804196,0.759259,0.732143,0.745455
2,0.809859,0.803922,0.706897,0.752294
3,0.866197,0.936170,0.733333,0.822430
4,0.802817,0.804348,0.660714,0.725490


In [24]:
metrics_mtx = metrics_df.to_numpy()
metrics_mtx.mean(axis=0)

array([0.81185856, 0.7903694 , 0.71245074, 0.74638855])

In [30]:
preparer = objects.datasets.TitanicDatasetPrepare(config.paths.titanic_train)

X, y = preparer.to_xy()
y

array([0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1,
       1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1,
       1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1,
       1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0,
       1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0,
       0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0,
       0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0,
       1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0,
       1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1,
       0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0,
       0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0,
       1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1,

### Models

In [ ]:
preparer = objects.datasets.TitanicDatasetPrepare(config.paths.titanic_train)

X, y = preparer.to_xy()

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, train_test_split
import numpy as np
from config import config
import objects.datasets

preparer = objects.datasets.TitanicDatasetPrepare(config.paths.titanic_train)

X, y = preparer.to_xy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=config.general.test_size, random_state=config.general.random_state)

log_reg = LogisticRegression(random_state=config.general.random_state)

params = {
    'C': np.linspace(1e-3, 5, 100),
    'penalty': ['l1', 'l2', 'elasticnet']
}

grid = GridSearchCV(log_reg, params, scoring='f1', verbose=True)
grid.fit(X_train, y_train)
grid.best_params_

Fitting 5 folds for each of 300 candidates, totalling 1500 fits


c:\programms\miniconda\envs\titanic-housing\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\programms\miniconda\envs\titanic-housing\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\programms\miniconda

{'C': np.float64(2.8287171717171713), 'penalty': 'l2'}

In [10]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
import numpy as np
from config import config
import objects.datasets

preparer = objects.datasets.TitanicDatasetPrepare(config.paths.titanic_train)

X, y = preparer.to_xy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=config.general.test_size, random_state=config.general.random_state)

knn = KNeighborsClassifier()

params = {
    'n_neighbors': [i for i in range(1, 16)],
    'weights': ['uniform', 'distance']
}

grid = GridSearchCV(knn, params, scoring='f1', verbose=True)
grid.fit(X_train, y_train)
grid.best_params_

Fitting 5 folds for each of 30 candidates, totalling 150 fits


{'n_neighbors': 5, 'weights': 'uniform'}

In [11]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
import numpy as np
from config import config
import objects.datasets

preparer = objects.datasets.TitanicDatasetPrepare(config.paths.titanic_train)

X, y = preparer.to_xy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=config.general.test_size, random_state=config.general.random_state, stratify=y)

tree = DecisionTreeClassifier(random_state=config.general.random_state)

params = {
    'criterion': ['gini', 'log_loss', 'entropy'],
    'min_samples_split': [i for i in range(2, 16)],
    'min_samples_leaf': [i for i in range(2, 16)],
    'min_impurity_decrease': np.linspace(0, 1, 10)
}

grid = GridSearchCV(tree, params, scoring='f1', verbose=True)
grid.fit(X_train, y_train)
grid.best_params_

Fitting 5 folds for each of 5880 candidates, totalling 29400 fits


{'criterion': 'log_loss',
 'min_impurity_decrease': np.float64(0.0),
 'min_samples_leaf': 3,
 'min_samples_split': 10}

In [14]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
import numpy as np
from config import config
import objects.datasets

preparer = objects.datasets.TitanicDatasetPrepare(config.paths.titanic_train)

X, y = preparer.to_xy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=config.general.test_size, random_state=config.general.random_state, stratify=y)

forest = RandomForestClassifier(random_state=config.general.random_state)

params = {
    'n_estimators': [i for i in range(10, 100, 10)],
    'criterion': ['log_loss'],
    'min_samples_split': [i for i in range(2, 16)],
    'min_samples_leaf': [i for i in range(2, 16)],
}

grid = GridSearchCV(forest, params, scoring='f1', verbose=True)
grid.fit(X_train, y_train)
grid.best_params_

Fitting 5 folds for each of 1764 candidates, totalling 8820 fits


{'criterion': 'log_loss',
 'min_samples_leaf': 3,
 'min_samples_split': 10,
 'n_estimators': 30}